# Penguin Slide

In [1]:
import numpy as np, random, time, pickle, os, threading
from IPython.display import display, clear_output
import ipywidgets as widgets
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:

class SlideEnv:
    def __init__(self, size=8, seed=None, holes=6, rocks=4):
        self.size = size
        self.holes = holes
        self.rocks = rocks
        self.rng = np.random.RandomState(seed if seed is not None else random.randint(0,999999))
        self.reset_grid()
        self.action_space = 4
        self.state = None

    def reset_grid(self):
        s = self.size
        self.grid = np.zeros((s, s), dtype=int)
        self.goal = (s - 1, s - 1)
        self.grid[self.goal] = 3
        holes = self.holes
        rocks = self.rocks


        placed = 0
        while placed < holes:
            i,j = self.rng.randint(0,s), self.rng.randint(0,s)
            if (i,j)!=self.goal and self.grid[i,j]==0:
                self.grid[i,j]=1; placed+=1

        placed=0
        while placed<rocks:
            i,j=self.rng.randint(0,s),self.rng.randint(0,s)
            if (i,j)!=self.goal and self.grid[i,j]==0:
                self.grid[i,j]=2; placed+=1

        self.start=(0,0)
        if self.grid[self.start]!=0:
            for i in range(s):
                for j in range(s):
                    if self.grid[i,j]==0:
                        self.start=(i,j);break
                else: continue
                break

        self.step_reward=-0.01
        self.hole_reward=-1.0
        self.goal_reward=1.0

    def reset(self):
        self.state=self.start
        return self.state_to_index(self.state)

    def state_to_index(self, state):
        i,j=state; return i*self.size+j

    def index_to_state(self, idx):
        return (idx//self.size, idx%self.size)

    def valid_move(self,state,action):
        i,j=state
        di=[-1,0,1,0]; dj=[0,1,0,-1]
        ni,nj=i+di[action], j+dj[action]
        if not (0<=ni<self.size and 0<=nj<self.size): return state
        if self.grid[ni,nj]==2: return state
        return (ni,nj)

    def step(self,action):
        next_state=self.valid_move(self.state,action)
        self.state=next_state
        t=self.grid[self.state]
        if t==1: return self.state_to_index(self.state), self.hole_reward, True
        if t==3: return self.state_to_index(self.state), self.goal_reward, True
        return self.state_to_index(self.state), self.step_reward, False


In [3]:

class RLAgent:
    def __init__(self, env, alpha=0.5, gamma=0.99, epsilon=1.0, min_epsilon=0.01, epsilon_decay=0.995):
        self.env = env
        self.n_states = env.size * env.size
        self.n_actions = env.action_space
        self.alpha, self.gamma = alpha, gamma
        self.epsilon, self.min_epsilon, self.epsilon_decay = epsilon, min_epsilon, epsilon_decay
        self.Q = np.zeros((self.n_states, self.n_actions), dtype=float)

    def choose_action(self, s):
        if random.random() < self.epsilon: return random.randrange(self.n_actions)
        return int(np.argmax(self.Q[s]))

    def decay(self):
        self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)

class QLearningAgent(RLAgent):
    def learn_episode(self, max_steps=200):
        s = self.env.reset(); total = 0.0
        for _ in range(max_steps):
            a = self.choose_action(s)
            ns, r, done = self.env.step(a)
            self.Q[s, a] += self.alpha * (r + self.gamma * np.max(self.Q[ns]) - self.Q[s, a])
            total += r; s = ns
            if done: break
        self.decay(); return total

class SARSAgent(RLAgent):
    def learn_episode(self, max_steps=200):
        s = self.env.reset(); a = self.choose_action(s); total = 0.0
        for _ in range(max_steps):
            ns, r, done = self.env.step(a)
            na = self.choose_action(ns)
            self.Q[s, a] += self.alpha * (r + self.gamma * self.Q[ns, na] - self.Q[s, a])
            total += r; s, a = ns, na
            if done: break
        self.decay(); return total

class ExpectedSarsaAgent(RLAgent):
    def learn_episode(self, max_steps=200):
        s = self.env.reset(); total = 0.0
        for _ in range(max_steps):
            a = self.choose_action(s)
            ns, r, done = self.env.step(a)
            greedy = int(np.argmax(self.Q[ns]))
            pi = np.ones(self.n_actions) * (self.epsilon / self.n_actions)
            pi[greedy] += (1.0 - self.epsilon)
            expQ = np.dot(self.Q[ns], pi)
            self.Q[s, a] += self.alpha * (r + self.gamma * expQ - self.Q[s, a])
            total += r; s = ns
            if done: break
        self.decay(); return total


In [4]:

def train_fast(agent, episodes=2000, max_steps=200):
    rewards = []
    for ep in range(episodes):
        rewards.append(agent.learn_episode(max_steps))
    return rewards


In [5]:

def make_animator(env, Q, cell_px=70, margin=8, fps=60):
    import pygame
    class Animator:
        def __init__(self, env, Q, cell_px=70, margin=8, fps=60):
            pygame.init()
            self.env, self.Q, self.size, self.cell, self.fps = env, Q, env.size, cell_px, fps
            w = self.cell * self.size + 2*margin
            h = self.cell * self.size + 2*margin
            self.screen = pygame.display.set_mode((w,h))
            pygame.display.set_caption('Penguin Slide — Learned Policy')
            self.clock = pygame.time.Clock()
            self.margin = margin
            self.font = pygame.font.SysFont(None, 28)
        def draw_grid(self):
            colors = {0:(245,255,250),1:(255,220,220),2:(120,120,120),3:(180,240,200)}
            for i in range(self.size):
                for j in range(self.size):
                    x = self.margin + j*self.cell
                    y = self.margin + i*self.cell
                    rect = pygame.Rect(x,y,self.cell,self.cell)
                    pygame.draw.rect(self.screen, colors[self.env.grid[i,j]], rect)
                    pygame.draw.rect(self.screen, (200,200,200), rect, 1)
        def agent_px(self, state):
            i,j = state
            x = self.margin + j*self.cell + self.cell//2
            y = self.margin + i*self.cell + self.cell//2
            return (x,y)
        def draw_penguin(self, pos):
            x, y = pos.astype(int)
            body_color = (30, 60, 150)
            belly_color = (255, 255, 255)
            beak_color = (255, 180, 0)
            pygame.draw.ellipse(self.screen, body_color, (x - self.cell//4, y - self.cell//2, self.cell//2, self.cell))
            pygame.draw.ellipse(self.screen, belly_color, (x - self.cell//6, y - self.cell//3, self.cell//3, int(self.cell/1.5)))
            pygame.draw.circle(self.screen, (255,255,255), (x - 6, y - self.cell//3), 4)
            pygame.draw.circle(self.screen, (255,255,255), (x + 6, y - self.cell//3), 4)
            pygame.draw.circle(self.screen, (0,0,0), (x - 6, y - self.cell//3), 2)
            pygame.draw.circle(self.screen, (0,0,0), (x + 6, y - self.cell//3), 2)
            pygame.draw.polygon(self.screen, beak_color, [(x, y - self.cell//4), (x - 5, y - self.cell//6), (x + 5, y - self.cell//6)])
        def animate_policy(self, speed=2.0):
            waiting = True
            while waiting:
                for event in pygame.event.get():
                    if event.type == pygame.QUIT:
                        pygame.quit(); return
                    if event.type == pygame.KEYDOWN or event.type == pygame.MOUSEBUTTONDOWN:
                        waiting = False
                self.screen.fill((230,245,255))
                msg = self.font.render('Press any key or click to start animation', True, (30,30,30))
                self.screen.blit(msg, (20, self.cell*self.size//2))
                pygame.display.flip(); self.clock.tick(30)
            s_idx = self.env.reset()
            s = self.env.index_to_state(s_idx)
            traj = [s]
            done = False; steps = 0
            while not done and steps < 200:
                a = int(np.argmax(self.Q[s_idx]))
                ns_idx, r, done = self.env.step(a)
                s_idx = ns_idx
                s = self.env.index_to_state(s_idx)
                traj.append(s)
                steps += 1
            for k in range(len(traj)-1):
                start = np.array(self.agent_px(traj[k]), dtype=float)
                end = np.array(self.agent_px(traj[k+1]), dtype=float)
                frames = max(1, int(self.fps * max(0.1, 1.0/speed)))
                for f in range(frames):
                    t = (f+1)/frames
                    pos = start*(1-t) + end*t
                    for event in pygame.event.get():
                        if event.type == pygame.QUIT:
                            pygame.quit(); return
                    self.screen.fill((230,245,255))
                    self.draw_grid()
                    self.draw_penguin(pos)
                    pygame.display.flip(); self.clock.tick(self.fps)
            pygame.time.wait(600)
    return Animator(env, Q, cell_px=cell_px, margin=margin, fps=fps)


In [6]:

# Widgets
alg_dd = widgets.Dropdown(options=[('Q-Learning','qlearning'), ('SARSA','sarsa'), ('Expected SARSA','expected_sarsa')], value='qlearning', description='Algorithm:')
episodes_slider = widgets.IntSlider(value=2000, min=100, max=20000, step=100, description='Episodes:')
speed_slider = widgets.FloatSlider(value=2.0, min=0.5, max=8.0, step=0.25, description='Speed:')
train_button = widgets.Button(description='Train', button_style='success')
anim_button = widgets.Button(description='Animate', button_style='info')
holes_slider = widgets.IntSlider(
    value=6, min=1, max=20, step=1, description='Holes:'
)

rocks_slider = widgets.IntSlider(
    value=4, min=1, max=20, step=1, description='Rocks:'
)

out = widgets.Output()
ui = widgets.VBox([
    alg_dd,
    episodes_slider,
    holes_slider,        # << NEW
    rocks_slider,        # << NEW
    speed_slider,
    train_button,
    anim_button
])

display(widgets.HBox([ui, out]))


In [7]:
GLOBAL = {}

def on_train(b):
    train_button.disabled = True
    out.clear_output()
    
    
    with out:
        print('Training...')

    alg = alg_dd.value
    episodes = episodes_slider.value
    seed = random.randint(0,999999)
    env = SlideEnv(seed=seed, holes=holes_slider.value, rocks=rocks_slider.value)


    # Choose agent
    if alg == 'qlearning':
        agent = QLearningAgent(env)
    elif alg == 'sarsa':
        agent = SARSAgent(env)
    else:
        agent = ExpectedSarsaAgent(env)

    # >>> RUN ON MAIN THREAD (NO THREADING)
    rewards = train_fast(agent, episodes=episodes)

    GLOBAL['env'] = env
    GLOBAL['agent'] = agent
    GLOBAL['rewards'] = rewards

    # save Q-table safely
    save_dir = os.path.join(os.getcwd(), 'trained_qtables')
    os.makedirs(save_dir, exist_ok=True)
    fname = os.path.join(save_dir, f'qtable_{alg}_{episodes}_{seed}.pkl')

    try:
        with open(fname, 'wb') as f:
            pickle.dump(agent.Q, f)
        GLOBAL['trained_Q_path'] = fname
    except Exception as e:
        with out:
            print('Could not save Q-table:', e)

    train_button.disabled = False

    # >>> NOW PLOT CORRECTLY
    with out:
        clear_output()
        print(f'Training finished on seed={seed}.')
        print(f"Last episode reward: {rewards[-1]:.3f}")

        # Convert to numpy for processing
        rewards_arr = np.array(rewards)

        # ---- NEW: Moving Average ----
        window = 100  # change if needed
        if len(rewards_arr) >= window:
            moving_avg = np.convolve(rewards_arr, np.ones(window)/window, mode='valid')
        else:
            moving_avg = rewards_arr  # fallback

        plt.figure(figsize=(8,4))

        # raw rewards
        plt.plot(rewards_arr, alpha=0.3, label="Reward per Episode")

        # moving average line
        plt.plot(range(window-1, len(moving_avg)+window-1), moving_avg, 
                linewidth=2, color='red', label=f"Moving Avg ({window})")

        plt.title("Rewards During Training")
        plt.xlabel("Episode")
        plt.ylabel("Reward")
        plt.grid(True)
        plt.legend()
        plt.show()



def on_anim(b):
    if 'agent' not in GLOBAL:
        with out:
            print('Train first!')
        return

    agent = GLOBAL['agent']
    env = GLOBAL['env']
    animator = make_animator(env, agent.Q, cell_px=70, margin=12, fps=60)
    animator.animate_policy(speed=speed_slider.value)


train_button.on_click(on_train)
anim_button.on_click(on_anim)
